# LeetCode 2264 — Largest 3-Same-Digit Number in String bằng Python

Link đề: https://leetcode.com/problems/largest-3-same-digit-number-in-string/ · Độ khó: **Easy** · Chủ đề: String

Skeleton LeetCode:

```python
class Solution:
    def largestGoodInteger(self, num: str) -> str:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu khái niệm **chuỗi con liên tiếp (substring)** độ dài cố định và cách duyệt bằng **cửa sổ trượt** (sliding window) kích thước `3`.
2. Biết **so sánh chuỗi theo thứ tự từ điển** và vì sao với các chuỗi cùng độ dài gồm chữ số, so sánh chuỗi cho cùng kết quả với so sánh số (kể cả `"000"`).
3. Dùng `max(..., default="")`, phép nhân chuỗi `"7" * 3`, toán tử `in`, và `itertools.groupby` để nhóm các chữ số giống nhau liên tiếp.
4. Viết 5 cách: thu thập mọi ứng viên rồi lấy `max`, quét một lần giữ kết quả tốt nhất, thử từng chữ số từ `9` đến `0`, `groupby` theo run, và biểu thức chính quy (regex).
5. Trace, test ngẫu nhiên đối chiếu, và tránh lỗi: đổi sang `int` làm mất số `0` đầu (`"000"`), quên trường hợp không có kết quả, dừng cửa sổ sai chỗ.

## 2. Đề bài & ví dụ

Cho chuỗi `num` biểu diễn một số nguyên lớn (chỉ gồm chữ số). Một số nguyên **tốt** (*good integer*) là chuỗi con của `num` thỏa **cả hai**:

- có **độ dài đúng 3**;
- gồm **ba chữ số giống nhau** (ví dụ `"777"`, `"000"`).

Trả về **số nguyên tốt lớn nhất** dưới dạng **chuỗi**. Nếu không có số tốt nào, trả về chuỗi rỗng `""`.

| Ví dụ | num | Output | Giải thích |
|---|---|---|---|
| 1 | `"6777133339"` | `"777"` | có `"777"` và `"333"`; `"777"` lớn hơn |
| 2 | `"2300019"` | `"000"` | chỉ có `"000"`; số có thể có chữ số `0` đứng đầu |
| 3 | `"42352338"` | `""` | không có ba chữ số giống nhau liên tiếp |

Constraints: `3 <= len(num) <= 1000`; `num` chỉ gồm chữ số.

Lưu ý: `num` có thể bắt đầu bằng `0` và kết quả `"000"` hợp lệ; vì vậy **phải giữ kiểu chuỗi** (nếu đổi sang `int`, `"000"` thành `0`).

In [1]:
examples = [("6777133339", "777"), ("2300019", "000"), ("42352338", "")]
for num, expected in examples:
    triples = [num[i:i + 3] for i in range(len(num) - 2)]
    good = [t for t in triples if t[0] == t[1] == t[2]]
    print(f"num={num!r:14} | các chuỗi con dài 3 giống nhau: {good} | mong đợi {expected!r}")

num='6777133339'   | các chuỗi con dài 3 giống nhau: ['777', '333', '333'] | mong đợi '777'
num='2300019'      | các chuỗi con dài 3 giống nhau: ['000'] | mong đợi '000'
num='42352338'     | các chuỗi con dài 3 giống nhau: [] | mong đợi ''


## 3. Kiến thức nền

### 3.1. Lát cắt chuỗi `num[i:i+3]`

`num[i:i+3]` lấy 3 ký tự bắt đầu từ chỉ số `i`. Để lát cắt **đủ 3 ký tự**, `i` chạy từ `0` đến `len(num) - 3` (gộp: `range(len(num) - 2)`). Nếu `i` quá gần cuối, lát cắt ngắn hơn 3.

In [2]:
num = "6777133339"
print([num[i:i + 3] for i in range(len(num) - 2)])
print(len(num), len(range(len(num) - 2)))
print(repr(num[8:11]), len(num[8:11]))      # i quá gần cuối: chỉ còn 2 ký tự

['677', '777', '771', '713', '133', '333', '333', '339']
10 8
'39' 2


### 3.2. So sánh ba ký tự bằng chuỗi `a == b == c`

Python cho phép **so sánh dây chuyền**: `a == b == c` đúng khi `a == b` **và** `b == c`. Vậy `num[i] == num[i+1] == num[i+2]` kiểm tra ba ký tự liên tiếp giống nhau.

In [3]:
print(1 == 1 == 1, 1 == 1 == 2)
t = "777"
print(t[0] == t[1] == t[2], "737"[0] == "737"[1] == "737"[2])

True False
True False


### 3.3. So sánh chuỗi theo thứ tự từ điển và nhân chuỗi

Python so sánh chuỗi **từng ký tự từ trái sang phải** theo mã ký tự. Các chữ số `'0' < '1' < ... < '9'`, nên với các chuỗi **cùng độ dài** gồm chữ số, so sánh chuỗi cho cùng thứ tự với so sánh số: `"777" > "333"`, `"333" > "000"`. Phép nhân chuỗi: `"7" * 3 == "777"`.

In [4]:
print("777" > "333", "333" > "000", "999" > "998")
print("7" * 3, "0" * 3, "9" * 3 == "999")
print(max(["333", "777", "000"]))

True True True
777 000 True
777


### 3.4. `max(iterable, default=...)`

`max([])` gây `ValueError`. Với tham số `default`, `max([], default="")` trả `""` khi dãy rỗng — rất tiện khi có thể không có ứng viên nào.

In [5]:
print(repr(max([], default="")))
print(max(["333", "777"], default=""))
try:
    max([])
except ValueError as e:
    print("lỗi:", e)

''
777
lỗi: max() iterable argument is empty


### 3.5. Toán tử `in` giữa hai chuỗi

`"777" in num` kiểm tra `"777"` có là **chuỗi con liên tiếp** của `num` không. Kết hợp với phép nhân chuỗi, ta kiểm tra ngay "có ba chữ số `d` liên tiếp không": `d * 3 in num`.

In [6]:
num = "6777133339"
for d in "9876543210":
    print(d, d * 3 in num, end=" | ")
print()
print(next((d * 3 for d in "9876543210" if d * 3 in num), ""))

9 False | 8 False | 7 True | 6 False | 5 False | 4 False | 3 True | 2 False | 1 False | 0 False | 
777


### 3.6. `itertools.groupby` — nhóm các phần tử giống nhau **liên tiếp**

`groupby("6777133339")` trả về các cặp `(ký tự, nhóm)` cho từng **đoạn** ký tự giống nhau kề nhau: `6`, `777`, `1`, `3333`... Độ dài mỗi đoạn ghi nhận bằng `len(list(group))`.

In [7]:
from itertools import groupby
runs = [(ch, len(list(g))) for ch, g in groupby("6777133339")]
print(runs)
print([(ch, n) for ch, n in runs if n >= 3])

[('6', 1), ('7', 3), ('1', 1), ('3', 4), ('9', 1)]
[('7', 3), ('3', 4)]


### 3.7. Biểu thức chính quy (regex) cơ bản

Mô-đun `re` tìm mẫu trong chuỗi. `(\d)\1\1` nghĩa: một chữ số (nhóm 1) rồi **lặp lại nhóm 1** thêm hai lần. Bài này cần các khớp **chồng lấp** (ví dụ `3333` chứa hai khớp bắt đầu ở vị trí khác nhau), nên dùng **lookahead** `(?=...)` để không "ăn" ký tự.

In [8]:
import re
print(re.findall(r"(\d)\1\1", "6777133339"))                 # chỉ chữ số lặp
print([m.group(1) * 3 for m in re.finditer(r"(?=((\d)\2\2))", "6777133339")])

['7', '3']
['777777777', '333333333', '333333333']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Lấy ví dụ 1: `num = "6777133339"`. Trượt một "ô cửa" rộng 3 qua chuỗi và xem nội dung từng vị trí:

| `i` | `num[i:i+3]` | ba ký tự giống nhau? |
|---|---|---|
| 0 | `677` | không |
| 1 | `777` | **có** |
| 2 | `771` | không |
| 3 | `713` | không |
| 4 | `133` | không |
| 5 | `333` | **có** |
| 6 | `333` | **có** |
| 7 | `339` | không |

Các ứng viên tốt: `777`, `333`, `333`. Lớn nhất là `777`. Nhớ rằng `3333` tạo **hai** lần `333` chồng lên nhau — nhưng chúng bằng nhau nên không ảnh hưởng đến `max`.

Vậy: **duyệt cửa sổ 3, giữ lại chuỗi giống nhau lớn nhất**.

In [9]:
num = "6777133339"
best = ""
for i in range(len(num) - 2):
    w = num[i:i + 3]
    same = w[0] == w[1] == w[2]
    if same and w > best:
        best = w
    print(f"i={i}: w={w!r} giống nhau={same!s:5} best={best!r}")

i=0: w='677' giống nhau=False best=''
i=1: w='777' giống nhau=True  best='777'
i=2: w='771' giống nhau=False best='777'
i=3: w='713' giống nhau=False best='777'
i=4: w='133' giống nhau=False best='777'
i=5: w='333' giống nhau=True  best='777'
i=6: w='333' giống nhau=True  best='777'
i=7: w='339' giống nhau=False best='777'


## 5. Cách 1 — Brute Force / baseline: thu thập mọi ứng viên rồi lấy `max`

Dựng danh sách mọi chuỗi con độ dài 3, lọc những chuỗi có ba ký tự giống nhau, rồi lấy `max` (hoặc `""` nếu rỗng). Dễ hiểu nhất vì tách hai việc "tìm" và "chọn lớn nhất".

In [10]:
def largest_good_collect(num):
    candidates = []
    for i in range(len(num) - 2):
        w = num[i:i + 3]
        if w[0] == w[1] == w[2]:
            candidates.append(w)
    return max(candidates) if candidates else ""

print(largest_good_collect("6777133339"), largest_good_collect("2300019"), repr(largest_good_collect("42352338")))

777 000 ''


In [11]:
from typing import List

class SolutionBrute:
    def largestGoodInteger(self, num: str) -> str:
        candidates = []
        for i in range(len(num) - 2):
            w = num[i:i + 3]
            if w[0] == w[1] == w[2]:
                candidates.append(w)
        return max(candidates) if candidates else ""

print(SolutionBrute().largestGoodInteger("6777133339"))
print(SolutionBrute().largestGoodInteger("2300019"))
print(repr(SolutionBrute().largestGoodInteger("42352338")))

777
000
''


**Độ phức tạp:** Gọi `n = len(num)`. Có `n - 2` cửa sổ, mỗi cửa sổ so sánh hằng số → **Time `O(n)`**; danh sách `candidates` tối đa `n - 2` phần tử → **Space `O(n)`**.

**Nút thắt:** lưu **mọi** ứng viên rồi mới chọn lớn nhất là thừa; ta chỉ cần giữ **ứng viên tốt nhất hiện tại** (Space `O(1)`).

## 6. Ý tưởng thuật toán / bước đột phá

Insight: số tốt chỉ có **10 khả năng**: `"000", "111", ..., "999"`. Và với hai chuỗi cùng độ dài gồm chữ số, so sánh chuỗi = so sánh số. Do đó:

- **quét một lần**, mỗi khi gặp ba ký tự giống nhau thì cập nhật `best = max(best, w)` — không cần lưu danh sách;
- hoặc **thử từ chữ số lớn nhất xuống**: nếu `"9"*3` có trong `num` thì trả về luôn; không thì thử `"8"*3`, ... — gặp cái đầu tiên là cái **lớn nhất**, có thể dừng sớm;
- chuỗi rỗng `""` nhỏ hơn mọi chuỗi không rỗng, nên khởi tạo `best = ""` cho phép so sánh `w > best` hoạt động ngay từ đầu mà không cần ngoại lệ.

## 7. Các cách tiếp theo

### Cách 2 — Cửa sổ trượt, giữ kết quả tốt nhất (`O(1)` bộ nhớ phụ)

Giống Cách 1 nhưng không lưu danh sách: chỉ giữ `best` và cập nhật khi cửa sổ giống nhau và lớn hơn `best`.

In [12]:
from typing import List

class SolutionWindow:
    def largestGoodInteger(self, num: str) -> str:
        best = ""
        for i in range(len(num) - 2):
            if num[i] == num[i + 1] == num[i + 2]:
                w = num[i:i + 3]
                if w > best:
                    best = w
        return best

print(SolutionWindow().largestGoodInteger("6777133339"))
print(SolutionWindow().largestGoodInteger("2300019"))
print(repr(SolutionWindow().largestGoodInteger("42352338")))

777
000
''


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (chuỗi `best` tối đa 3 ký tự). So sánh `w > best` giữa chuỗi cùng độ dài 3 là hằng số.

### Cách 3 — Thử từng chữ số từ `9` xuống `0`

Chỉ có 10 số tốt. Duyệt `d` từ `"9"` đến `"0"`; chữ số đầu tiên mà `d * 3 in num` cho đáp án (vì ta duyệt từ lớn đến nhỏ). Tối đa 10 lần kiểm tra `in`.

In [13]:
from typing import List

class SolutionDigits:
    def largestGoodInteger(self, num: str) -> str:
        for d in "9876543210":
            if d * 3 in num:
                return d * 3
        return ""

print(SolutionDigits().largestGoodInteger("6777133339"))
print(SolutionDigits().largestGoodInteger("2300019"))
print(repr(SolutionDigits().largestGoodInteger("42352338")))

777
000
''


**Độ phức tạp:** mỗi `d * 3 in num` tốn `O(n)`, tối đa 10 lần → **Time `O(10·n) = O(n)`**, **Space `O(1)`**. Rất gọn và dừng sớm khi tìm thấy; trường hợp xấu nhất khi không có số tốt nào là 10 lần quét.

### Cách 4 — `itertools.groupby` theo đoạn chữ số giống nhau

Chia `num` thành các **đoạn** ký tự giống nhau liên tiếp. Đoạn có độ dài `≥ 3` chứa số tốt `d * 3`. Lấy `max` của các `d * 3` đó. Cách này rõ ràng nếu bạn nghĩ theo "đoạn" thay vì "cửa sổ".

In [14]:
from itertools import groupby
from typing import List

class SolutionGroupBy:
    def largestGoodInteger(self, num: str) -> str:
        best = ""
        for ch, group in groupby(num):
            if len(list(group)) >= 3:
                best = max(best, ch * 3)
        return best

print(SolutionGroupBy().largestGoodInteger("6777133339"))
print(SolutionGroupBy().largestGoodInteger("2300019"))
print(repr(SolutionGroupBy().largestGoodInteger("42352338")))

777
000
''


**Độ phức tạp:** Time `O(n)`, Space `O(L)` với `L` là độ dài đoạn dài nhất do `list(group)` (có thể thay bằng `sum(1 for _ in group)` để còn `O(1)`).

### Cách 5 — Biểu thức chính quy với lookahead

`(?=((\d)\2\2))` khớp tại **mọi vị trí** mà ba ký tự tiếp theo là cùng một chữ số, mà không "ăn" ký tự — nên các khớp chồng lấp (như trong `3333`) đều tìm được. Ta lấy `max` các nhóm khớp.

In [15]:
import re
from typing import List

class SolutionRegex:
    def largestGoodInteger(self, num: str) -> str:
        found = [m.group(1) for m in re.finditer(r"(?=((\d)\2\2))", num)]
        return max(found) if found else ""

print(SolutionRegex().largestGoodInteger("6777133339"))
print(SolutionRegex().largestGoodInteger("2300019"))
print(repr(SolutionRegex().largestGoodInteger("42352338")))

777
000
''


**Độ phức tạp:** Time `O(n)` (mỗi vị trí kiểm tra hằng số), Space `O(k)` với `k` là số khớp. Regex ngắn nhưng khó đọc với người mới; hãy xem đây là công cụ bổ sung.

## 8. TRACE từng bước

Trace Cách 2 với `num = "2300019"`:

| `i` | `num[i:i+3]` | `num[i]==num[i+1]==num[i+2]`? | `best` |
|---|---|---|---|
| 0 | `230` | không | `""` |
| 1 | `300` | không | `""` |
| 2 | `000` | **có** → `"000" > ""` | `"000"` |
| 3 | `001` | không | `"000"` |
| 4 | `019` | không | `"000"` |

Kết quả `"000"`. Lưu ý `"000" > ""` đúng vì chuỗi rỗng nhỏ hơn mọi chuỗi không rỗng.

In [16]:
def trace_window(num):
    print(f"num={num!r}, n={len(num)}, số cửa sổ={len(num) - 2}")
    best = ""
    for i in range(len(num) - 2):
        w = num[i:i + 3]
        same = num[i] == num[i + 1] == num[i + 2]
        updated = same and w > best
        if updated:
            best = w
        print(f"  i={i}: w={w!r} same={same!s:5} cập nhật={updated!s:5} best={best!r}")
    return best

print("kết quả:", repr(trace_window("2300019")))
print("kết quả:", repr(trace_window("6777133339")))

num='2300019', n=7, số cửa sổ=5
  i=0: w='230' same=False cập nhật=False best=''
  i=1: w='300' same=False cập nhật=False best=''
  i=2: w='000' same=True  cập nhật=True  best='000'
  i=3: w='001' same=False cập nhật=False best='000'
  i=4: w='019' same=False cập nhật=False best='000'
kết quả: '000'
num='6777133339', n=10, số cửa sổ=8
  i=0: w='677' same=False cập nhật=False best=''
  i=1: w='777' same=True  cập nhật=True  best='777'
  i=2: w='771' same=False cập nhật=False best='777'
  i=3: w='713' same=False cập nhật=False best='777'
  i=4: w='133' same=False cập nhật=False best='777'
  i=5: w='333' same=True  cập nhật=False best='777'
  i=6: w='333' same=True  cập nhật=False best='777'
  i=7: w='339' same=False cập nhật=False best='777'
kết quả: '777'


In [17]:
def trace_digits(num):
    for d in "9876543210":
        found = d * 3 in num
        print(f"  thử {d*3!r}: có trong num? {found}")
        if found:
            return d * 3
    return ""

print("kết quả:", repr(trace_digits("6777133339")))
print("kết quả:", repr(trace_digits("42352338")))

  thử '999': có trong num? False
  thử '888': có trong num? False
  thử '777': có trong num? True
kết quả: '777'
  thử '999': có trong num? False
  thử '888': có trong num? False
  thử '777': có trong num? False
  thử '666': có trong num? False
  thử '555': có trong num? False
  thử '444': có trong num? False
  thử '333': có trong num? False
  thử '222': có trong num? False
  thử '111': có trong num? False
  thử '000': có trong num? False
kết quả: ''


## 9. Code nộp LeetCode

In [18]:
from typing import List

class Solution:
    def largestGoodInteger(self, num: str) -> str:
        for d in "9876543210":
            if d * 3 in num:
                return d * 3
        return ""

print(Solution().largestGoodInteger("6777133339"))
print(Solution().largestGoodInteger("2300019"))
print(repr(Solution().largestGoodInteger("42352338")))

777
000
''


## 10. Vì sao thuật toán đúng

- **Tập ứng viên đủ:** số tốt là chuỗi con dài 3 có ba ký tự giống nhau, tức dạng `"ddd"` với `d ∈ {0..9}`. Chuỗi `"ddd"` là chuỗi con của `num` ⇔ tồn tại `i` sao cho `num[i:i+3] == "ddd"` ⇔ `"ddd" in num`. Vậy `largest` = lớn nhất trong các `"ddd"` thuộc `num`.
- **So sánh chuỗi = so sánh số:** với hai chuỗi cùng độ dài 3 gồm chữ số, chữ số đầu khác nhau quyết định; vì `ddd` có cả ba chữ số bằng nhau nên `"ddd" < "eee"` ⇔ `d < e`. Do đó duyệt `d` từ `9` xuống `0` và dừng ở lần đầu thấy thì chắc chắn là lớn nhất (**invariant:** mọi chữ số lớn hơn `d` đã được kiểm tra và không có).
- **Không có kết quả:** nếu cả 10 phép kiểm tra đều `False`, không tồn tại số tốt, trả `""` đúng đề.
- **Cách cửa sổ:** `best` luôn là lớn nhất trong các ứng viên đã duyệt (khởi tạo `""` nhỏ hơn tất cả); sau khi duyệt hết `n - 2` cửa sổ, `best` là lớn nhất toàn chuỗi.
- **Giữ nguyên chữ số `0` đầu:** vì ta làm việc với chuỗi, `"000"` được trả về nguyên vẹn.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(num)` (`≤ 1000`).

| Cách | Time | Space |
|---|---|---|
| 1. Thu thập rồi `max` | `O(n)` | `O(n)` |
| 2. Cửa sổ trượt giữ `best` | `O(n)` | `O(1)` |
| 3. Thử `d * 3` từ 9 đến 0 | `O(10·n)` = `O(n)` | `O(1)` |
| 4. `groupby` | `O(n)` | `O(L)` (đoạn dài nhất) |
| 5. Regex lookahead | `O(n)` | `O(k)` (số khớp) |

Với `n ≤ 1000`, mọi cách chạy tức thời. Cách 3 chỉ có 10 lần `in` nên thực tế rất nhanh nhờ `in` của chuỗi được cài đặt bằng C.

## 12. Bộ test

Ví dụ đề, edge case (ngắn nhất 3 ký tự, `"000"` ở đầu, chuỗi `0000` chồng lấp, chuỗi rất dài, không có kết quả), phản ví dụ và test ngẫu nhiên đối chiếu **tất cả cách**.

In [19]:
import random

solvers = [SolutionBrute(), SolutionWindow(), SolutionDigits(), SolutionGroupBy(), SolutionRegex(), Solution()]

fixed = [
    ("6777133339", "777"),
    ("2300019", "000"),
    ("42352338", ""),
    ("111", "111"),                   # ngắn nhất, đúng ba ký tự
    ("123", ""),                      # ngắn nhất, không tốt
    ("000", "000"),                   # chỉ có số 0
    ("0000", "000"),                  # chồng lấp
    ("9999999", "999"),               # đoạn dài hơn 3
    ("1112223334445", "444"),         # nhiều ứng viên liền nhau, lớn nhất ở giữa
    ("777788", "777"),                # đoạn 4 chữ số 7: vẫn chỉ trả 777
    ("112233", ""),                   # từng đôi một, chưa đủ ba
    ("9" + "0" * 5 + "9", "000"),     # chữ số 9 ở hai đầu không tạo số tốt
]
for num, expected in fixed:
    for sol in solvers:
        assert sol.largestGoodInteger(num) == expected, (type(sol).__name__, num, sol.largestGoodInteger(num), expected)
print("test cố định: OK")

# chuỗi dài 1000: chỉ có "555" ở giữa, phần còn lại xen kẽ 1 và 2
long_num = "12" * 249 + "555" + "12" * 249 + "1"
assert len(long_num) == 1000
for sol in solvers:
    assert sol.largestGoodInteger(long_num) == "555"
print("chuỗi dài: OK, độ dài =", len(long_num))

def reference(num):
    best = ""
    for d in range(10):
        t = str(d) * 3
        for i in range(len(num) - 2):
            if num[i:i + 3] == t:
                best = max(best, t)
    return best

random.seed(2264)
for _ in range(600):
    n = random.randint(3, 25)
    alphabet = random.choice(["01", "012", "0123456789", "09"])
    num = "".join(random.choice(alphabet) for _ in range(n))
    expected = reference(num)
    for sol in solvers:
        assert sol.largestGoodInteger(num) == expected, (type(sol).__name__, num)
print("600 test ngẫu nhiên: OK")

test cố định: OK
chuỗi dài: OK, độ dài = 1000
600 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đổi sang `int` làm mất chữ số `0` đứng đầu

`int("000") == 0` và `str(0) == "0"`: không còn là `"000"`.

In [20]:
def wrong1(num):
    best = -1
    for i in range(len(num) - 2):
        if num[i] == num[i + 1] == num[i + 2]:
            best = max(best, int(num[i:i + 3]))
    return "" if best == -1 else str(best)

print("sai :", repr(wrong1("2300019")), "(đáng lẽ '000')")
print("đúng:", repr(Solution().largestGoodInteger("2300019")))

sai : '0' (đáng lẽ '000')
đúng: '000'


### Lỗi 2: Khởi tạo `best = "0"` hoặc `best = 0`

Khi không có số tốt, kết quả phải là `""`, không phải `"0"`. Và nếu `best = "000"` ban đầu thì luôn trả ít nhất `"000"`.

In [21]:
def wrong2(num):
    best = "000"
    for i in range(len(num) - 2):
        if num[i] == num[i + 1] == num[i + 2] and num[i:i + 3] > best:
            best = num[i:i + 3]
    return best

print("sai :", repr(wrong2("42352338")), "(đáng lẽ '')")
print("đúng:", repr(Solution().largestGoodInteger("42352338")))

sai : '000' (đáng lẽ '')
đúng: ''


### Lỗi 3: `range(len(num))` làm lát cắt cuối thiếu ký tự

Nếu duyệt `i` đến `len(num) - 1`, các lát cắt cuối có độ dài `< 3`; với cách `num[i+2]` còn gây `IndexError`.

In [22]:
def wrong3(num):
    best = ""
    for i in range(len(num)):                    # sai: phải là len(num) - 2
        if num[i] == num[i + 1] == num[i + 2]:
            best = max(best, num[i:i + 3])
    return best

try:
    wrong3("42352338")
except IndexError as e:
    print("sai -> IndexError:", e)
print("đúng:", repr(Solution().largestGoodInteger("42352338")))

sai -> IndexError: string index out of range
đúng: ''


### Lỗi 4: Chỉ kiểm tra hai ký tự giống nhau

`num[i] == num[i + 1]` mới là cặp đôi. Cần cả `num[i + 1] == num[i + 2]`.

In [23]:
def wrong4(num):
    best = ""
    for i in range(len(num) - 2):
        if num[i] == num[i + 1]:                 # thiếu so sánh thứ ba
            best = max(best, num[i:i + 3])
    return best

print("sai :", repr(wrong4("1123")), "| đúng:", repr(Solution().largestGoodInteger("1123")))

sai : '112' | đúng: ''


### Lỗi 5: Chọn ứng viên **đầu tiên** thay vì lớn nhất

`6777133339` có `"777"` trước `"333"`, nhưng với `"3337771"` ứng viên đầu tiên là `"333"` nhỏ hơn `"777"`.

In [24]:
def wrong5(num):
    for i in range(len(num) - 2):
        if num[i] == num[i + 1] == num[i + 2]:
            return num[i:i + 3]
    return ""

print("sai :", repr(wrong5("3337771")), "| đúng:", repr(Solution().largestGoodInteger("3337771")))

sai : '333' | đúng: '777'


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chuỗi con **độ dài cố định**" → **cửa sổ trượt**: `range(len(s) - k + 1)` và `s[i:i+k]`.
- Khi tập đáp án **nhỏ** (10 chữ số), có thể **liệt kê đáp án** thay vì quét: `d * 3 in num` từ lớn đến nhỏ.
- Đề trả về **chuỗi** có thể có số `0` đầu → giữ nguyên kiểu chuỗi, so sánh chuỗi cùng độ dài.
- "Lớn nhất hoặc rỗng nếu không có" → khởi tạo `best = ""` hoặc `max(..., default="")`.
- Đoạn các ký tự giống nhau liên tiếp → `itertools.groupby`.

## 15. Biến thể của bài

1. Chữ số lặp **`k`** lần thay vì 3 (tổng quát: `d * k in num`).
2. LeetCode 1446 Consecutive Characters: độ dài đoạn ký tự giống nhau dài nhất (`groupby`).
3. Trả về **vị trí** của số tốt lớn nhất, hoặc **đếm** số vị trí tốt.
4. Số tốt là chuỗi con 3 chữ số **liên tiếp tăng dần** (`"123"`, `"789"`).
5. Tìm chuỗi con dài nhất gồm toàn **một loại ký tự** (kỹ thuật hai con trỏ / sliding window mở rộng).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thu thập rồi `max` | lưu mọi ứng viên | `O(n)` | `O(n)` | rõ ràng nhất cho người mới |
| 2. Cửa sổ giữ `best` | quét một lần | `O(n)` | `O(1)` | cửa sổ độ dài `k` bất kỳ |
| 3. Thử `d*3` từ 9..0 | chỉ 10 ứng viên | `O(10n)` | `O(1)` | **khuyến nghị khi nộp**, ngắn và dừng sớm |
| 4. `groupby` | theo đoạn giống nhau | `O(n)` | `O(L)` | bài "đoạn liên tiếp" nói chung |
| 5. Regex lookahead | mẫu `(\d)\1\1` chồng lấp | `O(n)` | `O(k)` | bài mẫu phức tạp, xử lý văn bản |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `print("5" * 3, "ab" * 2, max(["333", "999", "000"]))` in ra gì?

**Bài 2 (tính tay).** `num = "3200014888"`. Tìm số tốt lớn nhất. Có những số tốt nào?

**Bài 3 (điền TODO).** Hoàn thành hàm trả về số tốt **nhỏ nhất** (hoặc `""` nếu không có).

In [25]:
def smallest_good(num):
    # TODO: dùng min với default hoặc thử d từ '0' đến '9'
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `largest_k_same(num, k)` trả về chuỗi con dài `k`, gồm `k` chữ số giống nhau, lớn nhất (hoặc `""`).

In [26]:
def largest_k_same(num, k):
    # TODO: tự viết, gợi ý d * k in num
    pass

**Bài 5 (mini challenge).** Viết `longest_run(num)` trả về `(chữ_số, độ_dài)` của **đoạn** chữ số giống nhau liên tiếp **dài nhất** (nếu hòa, lấy đoạn xuất hiện trước). Ví dụ `"6777133339"` → `("3", 4)`.

In [27]:
def longest_run(num):
    # TODO: dùng groupby hoặc hai con trỏ
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"5"*3 = "555"`, `"ab"*2 = "abab"`, `max` theo chuỗi là `"999"`. Bài 2: tìm các đoạn giống nhau từ 3 ký tự trở lên: `000` và `888`.
- **Mức 2 (vừa):** Bài 3: duyệt `d` từ `"0"` đến `"9"` và trả `d * 3` đầu tiên có trong `num`. Bài 4: tổng quát từ cách 3 với `d * k`.
- **Mức 3 (sâu):** Bài 5: `groupby` cho `(ch, len(list(g)))`; dùng `max(..., key=lambda t: t[1])` — `max` giữ phần tử **đầu tiên** khi hòa.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
from itertools import groupby

# Bài 1 và 2
assert ("5" * 3, "ab" * 2, max(["333", "999", "000"])) == ("555", "abab", "999")
assert Solution().largestGoodInteger("3200014888") == "888"

# Bài 3
def smallest_good(num):
    for d in "0123456789":
        if d * 3 in num:
            return d * 3
    return ""

# Bài 4
def largest_k_same(num, k):
    for d in "9876543210":
        if d * k in num:
            return d * k
    return ""

# Bài 5
def longest_run(num):
    runs = [(ch, len(list(g))) for ch, g in groupby(num)]
    return max(runs, key=lambda t: t[1])

assert smallest_good("3200014888") == "000"
assert smallest_good("123") == ""
assert largest_k_same("11222333", 2) == "33"
assert largest_k_same("11222333", 3) == "333"
assert largest_k_same("12345", 2) == ""
assert longest_run("6777133339") == ("3", 4)
assert longest_run("aabb") == ("a", 2)
assert longest_run("7") == ("7", 1)
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (thử từng chữ số lớn nhất xuống nhỏ nhất)

```python
class Solution:
    def largestGoodInteger(self, num: str) -> str:
        for d in "9876543210":
            if d * 3 in num:
                return d * 3
        return ""
```

**Checklist**

- [ ] Kết quả là **chuỗi**, không đổi sang `int` (giữ `"000"`)?
- [ ] Thử chữ số theo thứ tự **giảm dần** và trả về ngay lần đầu thấy?
- [ ] Trả `""` khi không có số tốt (không trả `"0"`)?
- [ ] Nếu dùng cửa sổ: `range(len(num) - 2)` và so sánh **cả ba** ký tự?
- [ ] Đã thử chuỗi ngắn nhất (`"111"`, `"123"`) và đoạn dài hơn 3 (`"9999999"`)?
- [ ] Đã thử phản ví dụ chọn "đầu tiên" thay vì "lớn nhất" (`"3337771"`)?
- [ ] Signature đúng `largestGoodInteger(self, num: str) -> str` và có `from typing import List`?